# 蔬菜批发价格波动检测与异常告警（数据分析案例）

> 数据：北京新发地批发市场公开行情（**真实数据**）
> 范围：2026-01-01 ~ 2026-08-11，145 个品种，22,362 条日度记录
> 完整脚本：`scripts/run_veg_monitor.py`；数据来源与清洗：`docs/DATA_SOURCES.md`

本 notebook 展示一条完整的数据分析链路：**数据获取 → 清洗质检 → 趋势与波动分析 →
异常检测 → 案例解读 → 业务建议**。所有路径均为相对路径，克隆仓库后可直接运行。


## 0. 准备工作

- 依赖：`pip install -r requirements.txt`
- 数据：仓库已内置清洗后的 `data/veg_wholesale_xinfadi.csv`；
  如需重新抓取，运行 `python scripts/fetch_xinfadi_data.py` + `python scripts/build_xinfadi_dataset.py`
- 检测逻辑复用 `scripts/run_veg_monitor.py` 中的函数，保证 notebook 与线上脚本一致


In [1]:
import sys
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd

# 中文字体（Windows）
mpl.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei"]
mpl.rcParams["axes.unicode_minus"] = False

# 项目根目录：兼容从 notebooks/ 或仓库根目录启动
CWD = Path.cwd()
ROOT = CWD.parent if CWD.name == "notebooks" else CWD
sys.path.insert(0, str(ROOT / "scripts"))

import run_veg_monitor as rvm

df = rvm.load_data(ROOT / "data" / "veg_wholesale_xinfadi.csv")
print(df.shape)
df.head()

(22362, 7)


,date,product,avg_price,low_price,high_price,n_records,origin
0,2026-01-01,丝瓜,5.15,4.5,5.8,1,豫冀
1,2026-01-02,丝瓜,5.05,4.5,5.6,1,豫冀
2,2026-01-03,丝瓜,5.05,4.5,5.6,1,豫冀
3,2026-01-04,丝瓜,5.05,4.5,5.6,1,豫冀
4,2026-01-05,丝瓜,5.05,4.5,5.6,1,豫冀


## 1. 数据质量与清洗

清洗规则（详见 `docs/DATA_SOURCES.md`）：

1. 只保留蔬菜类、以"斤"计价的记录；
2. 价格转数值，剔除缺失/非正记录；
3. 同一天同一品种的多个规格/产地记录聚合为一行（均价取均值、最低/最高取极值）；
4. 产地多值以 `/` 连接（如 `冀/辽`）。

下面检查关键质量指标。

In [2]:
# 1) 重复与价格合理性
dup = df.duplicated(["date", "product"]).sum()
print(f"重复 (date, product) 记录：{dup}")

# 2) 产地缺失
print(f"产地缺失占比：{df['origin'].eq('').mean() * 100:.1f}%")

# 3) 覆盖天数分布：多数品种全年覆盖，但仍有季节性品种不足 30 天
cov = df.groupby("product")["date"].nunique()
print(f"覆盖天数不足 30 天的品种数：{(cov < 30).sum()} / {len(cov)}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(cov, bins=30, color="#4C72B0", edgecolor="white")
axes[0].set_title("品种覆盖天数分布")
axes[0].set_xlabel("覆盖天数")
axes[0].set_ylabel("品种数")
axes[1].hist(df["avg_price"], bins=50, color="#55A868", edgecolor="white")
axes[1].set_title("均价分布（元/斤）")
axes[1].set_xlabel("均价")
fig.tight_layout()
fig.savefig(ROOT / "outputs" / "figures" / "eda_coverage.png", dpi=150, bbox_inches="tight")
print("已保存 outputs/figures/eda_coverage.png")

重复 (date, product) 记录：0
产地缺失占比：0.0%
覆盖天数不足 30 天的品种数：38 / 145


已保存 outputs/figures/eda_coverage.png


## 2. 价格走势总览

选取 12 个代表性品种（大路菜、果菜、叶菜、特色菜）观察整体走势。

可以看到：**叶菜类（油菜、菠菜、鸡毛菜、油麦菜）波动明显大于大路菜
（大白菜、土豆）**，且 7-8 月汛期波动加剧。

In [3]:
sample = ["大白菜", "黄瓜", "番茄", "土豆", "油菜", "油麦菜",
           "菠菜", "茄子", "冬瓜", "鸡毛菜", "茴香", "豌豆苗"]
fig, axes = plt.subplots(3, 4, figsize=(15, 9))
for ax, p in zip(axes.ravel(), sample):
    sel = df[df["product"] == p].sort_values("date")
    ax.plot(sel["date"], sel["avg_price"], lw=1.0, color="#4C72B0")
    ax.set_title(p, fontsize=11)
    ax.tick_params(labelsize=8)
    ax.set_ylabel("元/斤", fontsize=8)
fig.suptitle("12 个代表性品种的日度均价走势（2026-01 ~ 2026-08）", y=1.01)
fig.tight_layout()
fig.savefig(ROOT / "outputs" / "figures" / "eda_ts_sample.png", dpi=150, bbox_inches="tight")
print("已保存 outputs/figures/eda_ts_sample.png")

已保存 outputs/figures/eda_ts_sample.png


## 3. 波动与异常检测方法

| 信号 | 规则 |
|---|---|
| 价格异常 | 30 日滚动 z-score（|z| > 3） |
| 单日急涨 / 急跌 | 日环比 ≥ +15% / ≤ -12% |
| 产地切换异动 | 产地变化 且 价格变化 ≥ 5% |
| 持续高位 | z-score > 2 连续 ≥ 3 天 |
| 风险评分 | 波动率×10 + 价格异常×3 + 急涨 + 急跌 + 产地切换×2 + 持续高位×2 |

与 `scripts/run_veg_monitor.py` 完全同源，保证结果可复现。

In [4]:
v = rvm.compute_features(df)
v = rvm.classify_anomalies(v)
summary = rvm.build_summary(v)

print("=== 异常事件类型统计 ===")
for col, label in [("price_anom", "价格异常"), ("surge", "急涨"), ("crash", "急跌"),
                   ("origin_shift_alert", "产地切换异动"), ("sustained_high", "持续高位")]:
    print(f"{label}: {int(v[col].sum())}")
print(f"异常事件合计: {int(v['anomaly_any'].sum())}")

print("\n=== 月度异常分布 ===")
m = v.groupby(v["date"].dt.to_period("M")).agg(
    n_anomalies=("anomaly_any", "sum"), n_surge=("surge", "sum"), n_crash=("crash", "sum"))
m["month"] = m.index.astype(str)
print(m.reset_index(drop=True).to_string(index=False))

print("\n=== 风险评分 Top 10 ===")
print(summary.head(10).to_string(index=False))

=== 异常事件类型统计 ===
价格异常: 493
急涨: 790
急跌: 930
产地切换异动: 1721
持续高位: 505
异常事件合计: 3436

=== 月度异常分布 ===
 n_anomalies  n_surge  n_crash   month
         290       63       52 2026-01
         336       58       91 2026-02
         461       83      160 2026-03
         450       85      137 2026-04
         560      153      143 2026-05
         597      129      169 2026-06
         542      155      132 2026-07
         200       64       46 2026-08

=== 风险评分 Top 10 ===
product  n_days  mean_price  std_price  avg_rolling_vol  max_abs_z  n_price_anom  n_surge  n_crash  n_origin_shift  n_sustained  n_anomalies  anomaly_rate  risk_score
    鸡毛菜     153    3.931373   1.543821         0.140870   3.612797             3       19       21              74            9           77        0.5033      216.41
    蒿子秆     153    3.503157   1.097142         0.115621   3.436372             4       15       18              73            7           80        0.5229      206.16
     快菜     153    1.173203   0.

## 4. 案例解读（真实数据）

**① 春节效应**：2026-02 急跌 91 次（1 月仅 52 次），3 月急跌 160 次为全期最高——
节前备货推高价格（02-10 茴香 +25%、韭黄 +19%），节后需求回落引发集中下跌。

**② 汛期叶菜急跌**：2026-08-01~02 菠菜 -21.2%、油菜 -18.8%、小白菜 -36.4%，
夏季集中上市叠加降雨，叶菜价格大起大落。

**③ 产地切换伴随涨价**：2026-08-11 黄瓜产地切换且 +8.3%，产地切换往往伴随
货源与运费调整，是采购复核的重点信号。

**④ 持续高位**：油麦菜 8 月 11 日 4.75 元/斤创窗口新高（单日 +30.1%），
豌豆苗、散叶生菜连续多日 z-score > 2，处于统计意义上的历史高位。

In [5]:
# 重点案例可视化
cases = [("鸡毛菜", "叶菜·高波动代表"), ("油麦菜", "8月急涨+持续高位"),
         ("大白菜", "大路菜·基准对照"), ("黄瓜", "产地切换案例")]
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, (p, title) in zip(axes.ravel(), cases):
    sel = v[v["product"] == p].sort_values("date")
    ax.plot(sel["date"], sel["avg_price"], lw=1.2, color="#4C72B0", label="均价")
    ax.plot(sel["date"], sel["rolling_mean"], ls="--", lw=1, color="#DD8452", label="30日均值")
    for flag, color, marker, label in [("price_anom", "red", "o", "价格异常"),
                                       ("surge", "orange", "^", "急涨"),
                                       ("crash", "green", "v", "急跌")]:
        sub = sel[sel[flag]]
        if not sub.empty:
            ax.scatter(sub["date"], sub["avg_price"], c=color, marker=marker, s=16, label=label)
    ax.set_title(title)
    ax.legend(fontsize=7, ncol=3)
    ax.set_ylabel("元/斤", fontsize=8)
fig.suptitle("异常检测案例（红=价格异常，橙=急涨，绿=急跌）", y=1.01)
fig.tight_layout()
fig.savefig(ROOT / "outputs" / "figures" / "eda_cases.png", dpi=150, bbox_inches="tight")
print("已保存 outputs/figures/eda_cases.png")

已保存 outputs/figures/eda_cases.png


In [6]:
# 月度异常趋势图
m = v.groupby(v["date"].dt.to_period("M")).agg(
    n_anomalies=("anomaly_any", "sum"), n_surge=("surge", "sum"), n_crash=("crash", "sum"))
m["month"] = m.index.astype(str)

fig, ax = plt.subplots(figsize=(10, 3.8))
ax.bar(m["month"], m["n_anomalies"], color="#4C72B0", alpha=0.85, label="异常总数")
ax.plot(m["month"], m["n_surge"], marker="o", color="#F5A623", label="急涨")
ax.plot(m["month"], m["n_crash"], marker="s", color="#55A868", label="急跌")
ax.set_title("月度异常事件分布（2026-01 ~ 2026-08）")
ax.set_ylabel("事件数")
ax.legend()
fig.tight_layout()
fig.savefig(ROOT / "outputs" / "figures" / "eda_monthly.png", dpi=150, bbox_inches="tight")
print("已保存 outputs/figures/eda_monthly.png")

已保存 outputs/figures/eda_monthly.png


## 5. 结论与业务建议

**结论**

1. 波动风险高度集中在**叶菜类**：风险评分 Top10 中 9 个为叶菜（鸡毛菜、蒿子秆、快菜、
   奶白菜、盖菜、茼蒿、空心菜、苋菜、菊花菜），荠菜平均日波动率高达 36.5%；
2. **季节性规律清晰**：春节前后（1-3 月）先涨后跌，5-7 月夏季高温汛期波动最大
   （6 月异常 597 次为峰值）；
3. **产地切换是高频信号**：监测期内 1,721 次，占异常事件的一半，是采购复核的
   第一优先级信号。

**业务建议**

- 采购：叶菜类优先签订产地短约、锁定货源；急涨不追高、急跌控进货；
- 定价：价格异常确认 1-2 天后调价，排除单日噪音；
- 风控：产地切换 + 价格异动自动触发复核；春节前 2 周与 5-8 月汛期提高监控频率
  （每日自动运行：`scripts/schedule_windows.ps1`）；
- 扩展：接入多市场与成交量数据后，可恢复量价联动判别并引入季节性模型降低误报。

> 完整自动化产出见 `outputs/veg_report_index.html` 与 `outputs/business_summary.md`。
